# Option valuation

# MODELO DE COX-ROSS-RUBINSTEIN

In [30]:
import math
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['font.family'] = 'serif'

# Valor de la opción transcurrido un periodo

In [9]:
# Parámetros de una opción
S0 = 100.0  # Valor del activo subyascente
E = 100.0  # Precio de ejercicio
T = 1.0  # Vencimiento
r = 0.05  # Rentabilidad libre de riesgo
sigma = 0.2  # Volatilidad del activo subyascente
M=1  # Número de intervalos temporales

In [10]:
# Parámetros de la Binomial
dt=T/M  # Número de periodos
u = math.exp(sigma * math.sqrt(dt))  # incremento de precio
d = 1 / u  # reducción de precio
p = (math.exp(r * dt) - d) / (u - d)  # probabilidad de subida en un entorno libre de riesgo
u, d, p

(1.2214027581601699, 0.8187307530779818, 0.5774931963561243)

In [11]:
# Evolución de los precios del activo subyascente tras un periodo
Su=S0*u
Sd=S0*d
Su,Sd

(122.14027581601698, 81.87307530779819)

In [12]:
# Evolución el precio de la opción de compra
Cu=max(Su-E,0)
Cd=max(Sd-E,0)
Cu,Cd

(22.14027581601698, 0)

In [13]:
# Valor de la prima
C=math.exp(-r*T)*(p*Cu+(1-p)*Cd)
C

12.162284964623943

# Valoración de la opción con 2 periodos

In [14]:
# Evolución del precio del activo subyascente
Suu=Su*u
Sud=Su*d
Sdd=Sd*d
Suu,Sud,Sdd

(149.18246976412703, 100.0, 67.03200460356393)

In [15]:
# Valoración de la opción a vencimiento
Cuu=max(Suu-E,0)
Cud=max(Sud-E,0)
Cdd=max(Sdd-E,0)
Cuu,Cud,Cdd

(49.182469764127035, 0.0, 0)

In [16]:
# Valoración de la opción transcurrido un periodo
Cu=math.exp(-r*T)*(p*Cuu+(1-p)*Cud)
Cd=math.exp(-r*T)*(p*Cud+(1-p)*Cdd)
Cu,Cd

(27.017333365945593, 0.0)

In [17]:
# Valor de la prima
C=math.exp(-r*T)*(p*Cu+(1-p)*Cd)
C

14.84139177449445

# GENERALIZAMOS

El valor del activo financiero que se comporta como una binomial puede definirse como:
$$S_{ij}=S_0·u^{j-i}d^i$$

donde:
- $j$: es el número de pasos temporales
- $i$: es el número de bajadas acumuladas
- $j-i$: es el número de subidas acumuladas

Creamos la matriz de tiempos, donde cada columna representa un instante del tiempo:

$$mu=\left[\begin{array}{crrrl}
0 & 1 & 2 & \ldots & M  \\
0 & 1 & 2 &\ldots & M \\
\vdots & & & & \vdots \\
0 & 1 & 2 &\ldots & M \\
\end{array}\right]$$

np.tile(A, (r, c)) significa:

Repite la matriz A
- r veces en vertical (filas)
- c veces en horizontal (columnas)

In [18]:
M=2
# Array Initialization for Index Levels
j = np.arange(M + 1)               # vector intervalos de tiempo
mu = np.tile(j, (M + 1, 1))        # matriz temporal donde cada columna es un instante de tiempo
mu

array([[0, 1, 2],
       [0, 1, 2],
       [0, 1, 2]])

Creamos una segunda matriz de caidas donde cada fila representa el número de caídas acumuladas:

$$md=\left[\begin{array}{crrrl}
0 & 0 & 0 & \ldots & 0  \\
1 & 1 & 1 &\ldots & 1 \\
\vdots & & & & \vdots \\
M & M & M &\ldots & M \\
\end{array}\right]$$

In [19]:
md = mu.T
md

array([[0, 0, 0],
       [1, 1, 1],
       [2, 2, 2]])

Podemos estimar el número de subidas de la siguiente manera:

$$mu-md=
\begin{pmatrix}
0-0&1-0&2-0&\ldots&M-0\\
0-1&1-1&2-1&\ldots&M-1\\
\vdots& & & &\vdots\\
0-M&1-M&2-M&\ldots&M-M
\end{pmatrix}
=
\begin{pmatrix}
0&1&2&\ldots&M\\
-1&0&1& \ldots&M-1\\
\vdots& & & &\vdots\\
-M& 1-M& 2-M&\ldots&0
\end{pmatrix}$$


En un árbol binomial real, solo tiene sentido el triángulo donde $i\le j$.

En la matriz completa, cuando $i>j$ aparece $j-i<0$ (exponente negativo), que no corresponde a un nodo válido del árbol. Por ello introducimos un filtro.





In [20]:
# máscara del triángulo válido: i <= j
valid = md <= mu

In [21]:
# matriz de precios inicializada a NaN
S = np.full((M + 1, M + 1), np.nan, dtype=float)
S

array([[nan, nan, nan],
       [nan, nan, nan],
       [nan, nan, nan]])

In [22]:
# solo calculamos donde el nodo existe
S[valid] = S0 * (u ** (mu[valid] - md[valid])) * (d ** md[valid])
S

array([[100.        , 122.14027582, 149.18246976],
       [         nan,  81.87307531, 100.        ],
       [         nan,          nan,  67.0320046 ]])

In [23]:
def binomial_tree_prices(S0, u, d, M):
    """
    Construye la matriz de precios S[i,j] = S0 * u^(j-i) * d^i
    pero solo para nodos válidos (i <= j). El resto -> NaN.

    i = nº de bajadas (fila)
    j = paso temporal (columna)
    """
    j = np.arange(M + 1)                      # 0..M
    mu = np.tile(j, (M + 1, 1))               # mu[i,j] = j
    md = mu.T                                 # md[i,j] = i

    # máscara del triángulo válido: i <= j
    valid = md <= mu

    # matriz de precios inicializada a NaN
    S = np.full((M + 1, M + 1), np.nan, dtype=float)

    # solo calculamos donde el nodo existe
    S[valid] = S0 * (u ** (mu[valid] - md[valid])) * (d ** md[valid])

    return S

In [24]:
binomial_tree_prices(S0, u, d, M)

array([[100.        , 122.14027582, 149.18246976],
       [         nan,  81.87307531, 100.        ],
       [         nan,          nan,  67.0320046 ]])

In [25]:
V = np.maximum(S - E, 0)  # inner values for European call option
V

array([[ 0.        , 22.14027582, 49.18246976],
       [        nan,  0.        ,  0.        ],
       [        nan,         nan,  0.        ]])

In [26]:
df = math.exp(-r * dt)  # discount per interval
z = 0
for t in range(M - 1, -1, -1):  # backwards iteration
  V[0:M - z, t] = (p* V[0:M - z, t + 1]+ (1 - p) * V[1:M - z + 1, t + 1]) * df
  z += 1
#V[0, 0]
V

array([[14.84139177, 27.01733337, 49.18246976],
       [        nan,  0.        ,  0.        ],
       [        nan,         nan,  0.        ]])

In [27]:
def CRR_option_value(S0, E, T, r, sigma, otype, M=4):
    ''' Cox-Ross-Rubinstein European option valuation.

    Parameters
    ==========
    S0: float
        stock/index level at time 0
    E:  float
        strike price
    T:  float
        date of maturity
    r:  float
        constant, risk-less short rate
    sigma:  float
        volatility
    otype:  string
        either 'call' or 'put'
    M:  int
        number of time intervals
    '''
    # Time Parameters
    dt = T / M  # length of time interval
    df = math.exp(-r * dt)  # discount per interval

    # Binomial Parameters
    u = math.exp(sigma * math.sqrt(dt))  # up movement
    d = 1 / u  # down movement
    q = (math.exp(r * dt) - d) / (u - d)  # martingale branch probability

    # Array Initialization for Index Levels
    binomial_tree_prices(S0, u, d, M)

    # Inner Values
    if otype == 'call':
        V = np.maximum(S - E, 0)  # inner values for European call option
    else:
        V = np.maximum(E - S, 0)  # inner values for European put option

    z = 0
    for t in range(M - 1, -1, -1):  # backwards iteration
        V[0:M - z, t] = (q * V[0:M - z, t + 1]
                        + (1 - q) * V[1:M - z + 1, t + 1]) * df
        z += 1
    return V

In [28]:
V

array([[14.84139177, 27.01733337, 49.18246976],
       [        nan,  0.        ,  0.        ],
       [        nan,         nan,  0.        ]])

In [29]:
def plot_convergence(mmin, mmax, step_size):
    ''' Plots the CRR option values for increasing number of time
    intervals M against the Black-Scholes-Merton benchmark value.'''
    BSM_benchmark = BSM_call_value(S0, K, 0, T, r, sigma)
    m = xrange(mmin, mmax, step_size)
    CRR_values = [CRR_option_value(S0, K, T, r, sigma, 'call', M) for M in m]
    plt.figure(figsize=(9, 5))
    plt.plot(m, CRR_values, label='CRR values')
    plt.axhline(BSM_benchmark, color='r', ls='dashed', lw=1.5,
                label='BSM benchmark')
    plt.grid()
    plt.xlabel('# of binomial steps $M$')
    plt.ylabel('European call option value')
    plt.legend(loc=4)
    plt.xlim(0, mmax)